# 🦀 Day 3: Lifetimes in Structs

Today: Structs that hold references!

---

In [ ]:
// A struct that holds a reference MUST have a lifetime annotation

#[derive(Debug)]
struct Excerpt<'a> {
    text: &'a str,
}

// Means: the Excerpt cannot outlive the data it references
let novel = String::from("Call me Ishmael. Some years ago...");
let first_sentence;
{
    let i = novel.find('.').unwrap_or(novel.len());
    first_sentence = Excerpt { text: &novel[..i] };
}
println!("Excerpt: {:?}", first_sentence); // ✅ novel is still alive

In [ ]:
// Methods on structs with lifetimes
#[derive(Debug)]
struct TextWindow<'a> {
    full_text: &'a str,
    start: usize,
    end: usize,
}

impl<'a> TextWindow<'a> {
    fn new(text: &'a str) -> Self {
        TextWindow { full_text: text, start: 0, end: text.len() }
    }

    fn visible(&self) -> &str {
        &self.full_text[self.start..self.end]
    }

    fn narrow(&mut self, by: usize) {
        self.start = (self.start + by).min(self.end);
        self.end = self.end.saturating_sub(by);
    }

    fn word_at(&self, index: usize) -> Option<&str> {
        self.visible().split_whitespace().nth(index)
    }
}

let text = String::from("The quick brown fox jumps over the lazy dog");
let mut window = TextWindow::new(&text);
println!("Full: '{}'", window.visible());
window.narrow(4);
println!("Narrowed: '{}'", window.visible());
println!("Word 0: {:?}", window.word_at(0));

In [ ]:
// Multiple lifetimes in a struct
#[derive(Debug)]
struct Comparison<'a, 'b> {
    original: &'a str,
    modified: &'b str,
}

impl<'a, 'b> Comparison<'a, 'b> {
    fn differences(&self) -> Vec<(usize, char, char)> {
        self.original.chars().zip(self.modified.chars())
            .enumerate()
            .filter(|(_, (a, b))| a != b)
            .map(|(i, (a, b))| (i, a, b))
            .collect()
    }
}

let orig = "hello world";
let modified = "Hello World";
let comp = Comparison { original: orig, modified: modified };
for (i, a, b) in comp.differences() {
    println!("Pos {}: '{}' → '{}'", i, a, b);
}

In [ ]:
// Mixing owned and borrowed data
#[derive(Debug)]
struct Config<'a> {
    name: String,        // Owned — struct manages this
    template: &'a str,   // Borrowed — struct doesn't own this
    max_retries: u32,    // Copy type
}

impl<'a> Config<'a> {
    fn new(name: &str, template: &'a str) -> Self {
        Config {
            name: name.to_string(),  // Clone into owned String
            template,
            max_retries: 3,
        }
    }

    fn render(&self, value: &str) -> String {
        self.template.replace("{}", value)
    }
}

let template = "Hello, {}! Welcome to the system.";
let config = Config::new("Greeter", template);
println!("{}", config.render("Alice"));
println!("Config: {:?}", config);

## 🏋️ Exercises

In [ ]:
// Exercise 1: Create a struct `Highlight<'a>` that holds
// a reference to text and a start/end position.
// Add methods: highlighted_text(), before(), after()

// YOUR CODE HERE


In [ ]:
// Exercise 2: Create a struct `CsvRow<'a>` that holds
// references to column headers and cell values (both &str slices).
// Add a method get(column_name) -> Option<&str>

// YOUR CODE HERE


## 🎯 Key Takeaways

1. Structs holding references **must** declare lifetime parameters
2. `struct Foo<'a>` means "Foo can't outlive the data it borrows"
3. Use multiple lifetimes when fields borrow from different sources
4. Mix owned (`String`) and borrowed (`&str`) fields freely
5. `impl<'a> Foo<'a>` — lifetime goes on both impl and type

---
**Tomorrow:** `'static` lifetime! 🔒